# TREC 2023 External Evaluation — FROZEN pipeline, one shot

Runs the frozen ctmatch pipeline (developed on 2021/2022) on **TREC 2023 Clinical Trials** — a
genuinely held-out external test (no component ever saw it). **Nothing is trained or tuned here.**

Prereqs: run `build_corpus_2023.ipynb` first (produces the 2023 corpus + questionnaire→text + qrels).
Uses the frozen `ensemble_full_v1.txt` LambdaMART, `clf-v4`, `reranker-v2`, `retriever-v2`, Qwen-2.5-7B.

**Honest caveat:** 2023 topics are questionnaire-format (domain shift from the vignettes we're tuned
for) and the corpus is a different snapshot. This tests *generalization*; a strong result is a strong
claim, a weak one is confounded by the format shift. Report once — no iterating on 2023.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval rank-bm25 sentence-transformers datasets transformers accelerate tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
T23       = f'{DATA_ROOT}/trec2023'
CORPUS    = f'{T23}/doc_texts_fulltext_2023.txt'
INDEX     = f'{T23}/index2docid_2023.txt'
TOPICS    = f'{T23}/topics2023_text.jsonl'
QRELS     = f'{T23}/qrels2023.txt'
DENSE_EMB = f'{T23}/doc_embeddings_2023_retriever-v2.npy'   # encoded here (cached)
BM25_CACHE = f'{T23}/bm25_2023.pkl'
LLM_SCORES = f'{T23}/llm_scores_2023.jsonl'

CLF_CKPT  = 'semaj83/ctmatch-clf-v4'
V2_CKPT   = f'{DATA_ROOT}/reranker_hardneg_v2'
RETRIEVER = 'semaj83/ctmatch-retriever-v2'
LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
ENSEMBLE  = f'{DATA_ROOT}/ensemble_full_v1.txt'   # FROZEN LambdaMART

CAND_K, LLM_TOP_K, LLM_FLOOR, DOC_CHARS = 1000, 500, None, 1800
FEATURES = ['bm25','bm25_rank','dense','dense_rank','rrf','clf_rel','clf_partial','v2_rel','llm_yesno','llm_scored']
print('config set')

In [ ]:
import json, numpy as np
corpus_ids = [l.strip() for l in open(INDEX)]
corpus_txt = [l.rstrip('\n') for l in open(CORPUS)]
assert len(corpus_ids) == len(corpus_txt)
id2txt = dict(zip(corpus_ids, corpus_txt)); id2idx = {d: i for i, d in enumerate(corpus_ids)}
topics = {r['topic_id']: r['topic_text'] for r in map(json.loads, open(TOPICS))}
rel = {}
for l in open(QRELS):
    t, _, d, r = l.split(); rel.setdefault(t, {})[d] = int(r)
topics = {t: x for t, x in topics.items() if t in rel}   # keep judged topics
print(f'corpus {len(corpus_ids):,} | judged topics {len(topics)}')

In [ ]:
import pickle, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# BM25 (cache)
if os.path.exists(BM25_CACHE):
    bm25 = pickle.load(open(BM25_CACHE, 'rb'))
else:
    bm25 = BM25Okapi([t.lower().split() for t in tqdm(corpus_txt, desc='bm25')])
    pickle.dump(bm25, open(BM25_CACHE, 'wb'))

# dense: encode the 2023 corpus with retriever-v2 (~1 hr GPU; cached)
enc = SentenceTransformer(RETRIEVER)
if os.path.exists(DENSE_EMB):
    doc_emb = np.load(DENSE_EMB).astype(np.float32)
else:
    doc_emb = enc.encode(corpus_txt, convert_to_numpy=True, normalize_embeddings=True,
                         batch_size=256, show_progress_bar=True).astype(np.float32)
    np.save(DENSE_EMB, doc_emb)
doc_emb /= (np.linalg.norm(doc_emb, axis=1, keepdims=True) + 1e-9)
print('retrieval ready', doc_emb.shape)

In [ ]:
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
clf_tok = AutoTokenizer.from_pretrained(CLF_CKPT)
clf = AutoModelForSequenceClassification.from_pretrained(CLF_CKPT).to(device).eval()
v2  = AutoModelForSequenceClassification.from_pretrained(V2_CKPT).to(device).eval()
_ix = lambda mdl, nm: next((int(k) for k, v in mdl.config.id2label.items() if nm in v.lower()), None)
CLF_REL, CLF_PAR, V2_REL = _ix(clf,'relevant'), _ix(clf,'partial'), _ix(v2,'relevant')
def ce(text, texts, mdl, ri, pi, b=64):
    R,P=[],[]
    for i in range(0,len(texts),b):
        c=texts[i:i+b]; e=clf_tok([text]*len(c),c,padding=True,truncation=True,max_length=512,return_tensors='pt').to(device)
        with torch.no_grad(): p=F.softmax(mdl(**e).logits,-1).cpu().numpy()
        R+=list(p[:,ri]); P+=list(p[:,pi] if pi is not None else [0.0]*len(c))
    return R,P
ltok = AutoTokenizer.from_pretrained(LLM_MODEL, padding_side='left'); ltok.pad_token=ltok.pad_token or ltok.eos_token
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map='auto').eval()
YES=sorted({ltok.encode(w,add_special_tokens=False)[0] for w in ['yes','Yes',' yes',' Yes']})
NO =sorted({ltok.encode(w,add_special_tokens=False)[0] for w in ['no','No',' no',' No']})
def lprompt(pt,tr): return ltok.apply_chat_template([{'role':'system','content':'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'},{'role':'user','content':f'You are a clinical trial matching expert.\n\nPatient:\n{pt}\n\nTrial:\n{tr[:DOC_CHARS]}\n\nIs this patient likely eligible for this trial? Answer with a single word: yes or no.'}],tokenize=False,add_generation_prompt=True)
@torch.no_grad()
def lscore(pt,trs,b=8):
    o=[]
    for i in range(0,len(trs),b):
        e=ltok([lprompt(pt,t) for t in trs[i:i+b]],return_tensors='pt',padding=True,truncation=True,max_length=2048).to(llm.device)
        lg=llm(**e).logits[:,-1,:].float(); o+=(torch.logsumexp(lg[:,YES],-1)-torch.logsumexp(lg[:,NO],-1)).cpu().tolist()
    return o
print('models ready | clf', CLF_REL, CLF_PAR, '| v2', V2_REL)

In [ ]:
# Build the candidate pool + 9 features per topic, then score with the FROZEN ensemble.
import lightgbm as lgb
booster = lgb.Booster(model_file=ENSEMBLE)

# resume LLM scores
llm_done = {}
if os.path.exists(LLM_SCORES):
    for l in open(LLM_SCORES):
        r=json.loads(l); llm_done[(r['topic_id'],r['doc_id'])]=r['llm_score']

run = {}
lf = open(LLM_SCORES,'a')
for tid, text in tqdm(topics.items(), desc='topics'):
    qv = enc.encode(text, normalize_embeddings=True).astype(np.float32)
    bm = np.array(bm25.get_scores(text.lower().split())); dn = doc_emb @ qv
    bmT = np.argpartition(-bm, CAND_K)[:CAND_K]; dnT = np.argpartition(-dn, CAND_K)[:CAND_K]
    cand = list(set(bmT.tolist()) | set(dnT.tolist()))
    bmr = {j:r for r,j in enumerate(sorted(cand,key=lambda j:-bm[j]))}
    dnr = {j:r for r,j in enumerate(sorted(cand,key=lambda j:-dn[j]))}
    rrf = {j: 1/(60+bmr[j]+1)+1/(60+dnr[j]+1) for j in cand}
    texts = [corpus_txt[j] for j in cand]
    crel,cpar = ce(text,texts,clf,CLF_REL,CLF_PAR); vrel,_ = ce(text,texts,v2,V2_REL,None)
    top = [j for j,_ in sorted(rrf.items(),key=lambda x:-x[1])[:LLM_TOP_K]]
    need = [j for j in top if (tid,corpus_ids[j]) not in llm_done]
    if need:
        for j,s in zip(need, lscore(text,[corpus_txt[j] for j in need])):
            llm_done[(tid,corpus_ids[j])]=s; lf.write(json.dumps({'topic_id':tid,'doc_id':corpus_ids[j],'llm_score':float(s)})+'\n')
        lf.flush()
    floor = min(llm_done.values())-5.0
    X=[]
    for k,j in enumerate(cand):
        lv = llm_done.get((tid,corpus_ids[j]))
        X.append([bm[j],bmr[j],float(dn[j]),dnr[j],rrf[j],crel[k],cpar[k],vrel[k],
                  lv if lv is not None else floor, 1.0 if lv is not None else 0.0])
    sc = booster.predict(np.array(X,dtype=np.float32))
    run[tid] = {corpus_ids[j]: float(s) for j,s in zip(cand,sc)}
lf.close(); print('scored', len(run), 'topics')

In [ ]:
import pytrec_eval
qrel = {t: {d:int(r) for d,r in dd.items()} for t,dd in rel.items() if t in run}
def score(q):
    e=pytrec_eval.RelevanceEvaluator(q,{'ndcg_cut.10','P.10','recip_rank','recall.1000'}).evaluate(run)
    return {m:np.mean([v[m] for v in e.values()]) for m in ['ndcg_cut_10','P_10','recip_rank','recall_1000']}
graded = score(qrel)
elig = score({t:{d:(1 if r==2 else 0) for d,r in dd.items()} for t,dd in qrel.items()})
print('=== TREC 2023 (external, frozen pipeline) — one-shot ===')
print(f"  NDCG@10 (graded)      : {graded['ndcg_cut_10']:.4f}")
print(f"  P@10 (eligible-only)  : {elig['P_10']:.4f}")
print(f"  MRR  (eligible-only)  : {elig['recip_rank']:.4f}")
print(f"  Recall@1000           : {graded['recall_1000']:.4f}")
print('\nReference: our TREC22 NDCG@10=0.6105; TREC22 winner 0.6125. (2023 best-run: fill from overview.)')
print('Interpret: 2023 is questionnaire-format (domain shift) — read as a generalization test.')
# per-topic for a paired bootstrap vs any baseline run you obtain
pt = pytrec_eval.RelevanceEvaluator(qrel,{'ndcg_cut.10'}).evaluate(run)
per_topic = np.array([pt[t]['ndcg_cut_10'] for t in run])
boot=[np.random.default_rng(0).choice(per_topic,len(per_topic),replace=True).mean() for _ in range(10000)]
print(f"  bootstrap 95% CI: [{np.percentile(boot,2.5):.4f}, {np.percentile(boot,97.5):.4f}]  (n={len(per_topic)})")